In [2]:
# from dotenv import load_dotenv
# import json
# load_dotenv("../.env")
# import dspy

In [ ]:
# DON'T RUN THIS NOTEBOOK, for reference only
1/0

setup dspy

In [2]:
dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)

In [3]:
lm = dspy.LM(
    "openai/gpt-5-2025-08-07",
    model_type="responses",
    temperature=1.0,
    max_tokens=32000,
    reasoning={
        "effort": 'medium',
        "summary": 'detailed'
    },
    # service_tier="flex",
    # timeout=900
)
dspy.configure(lm=lm, adapter=dspy.JSONAdapter())

assemble full dataset

In [7]:
dataset = []

In [8]:
from pathlib import Path

base_path = Path("/Users/saner/game-arena/game-arena/aws-logs/final/four_in_a_row-fen/")

# Loop over all game_log.json files in the directory tree
for game_log_path in base_path.rglob("game_log.json"):
    try:
        with open(game_log_path, "r") as f:
            game = json.load(f)
        
        # Get the relative path for recording
        relative_path = str(game_log_path.relative_to(base_path))
        
        for index, event in enumerate(game.get('events', [])):
            if event.get('type') == 'turn':
                if 'generation' in event and 'reasoning' in event['generation'] and 'output' in event['generation']:
                    dataset.append({
                        "game_path": relative_path,
                        "turn_idx": index,
                        "player_id": event.get("player"),
                        "player": game["players"][event.get("player")],
                        "trace": event['generation']['reasoning'] + event['generation']['output'],
                    })
    except Exception as e:
        print(f"Error processing {game_log_path}: {e}")
        continue

print(f"Loaded {len(dataset)} turns from {len(list(base_path.rglob('game_log.json')))} games")

Loaded 18888 turns from 1404 games


In [6]:
loaded_optimized_program = dspy.load("artifacts/judge_gepa_light_program/")

In [7]:
loaded_optimized_program

extract_move_trees_module = Predict(StringSignature(trace -> trees
    instructions='You are given a natural-language reasoning trace about a 4×9 Four-in-a-Row (Connect-X-like) game. Your task is to reconstruct exactly the explicitly considered move trees and output strict JSON in a prescribed nested array format.\n\nDomain and board facts (authoritative; use as-is, do not infer/alter):\n- Board: 4 rows × 9 columns, zero-based. Rows 0–3 (top to bottom), columns 0–8 (left to right).\n- No gravity: a move may be played in any empty cell. Ignore any “falling piece” assumptions.\n- Win: four in a row horizontally, vertically, or diagonally (both down-right and down-left).\n- Vertical four is exactly a whole column (rows 0–3 in that column).\n- Diagonals of length 4 are exactly:\n  - Down-right (top-left → bottom-right): start only at row 0, columns 0–5. Each is (0,c),(1,c+1),(2,c+2),(3,c+3).\n  - Down-left (top-right → bottom-left): start only at row 0, columns 3–8. Each is (0,c),(1,c-1),(

extract_move_trees_module = Predict(StringSignature(trace -> trees
    instructions='You are given a natural-language reasoning trace about a 4×9 Four-in-a-Row (Connect-X-like) game. Your task is to reconstruct exactly the explicitly considered move trees and output strict JSON in a prescribed nested array format.\n\nDomain and board facts (authoritative; use as-is, do not infer/alter):\n- Board: 4 rows × 9 columns, zero-based. Rows 0–3 (top to bottom), columns 0–8 (left to right).\n- No gravity: a move may be played in any empty cell. Ignore any “falling piece” assumptions.\n- Win: four in a row horizontally, vertically, or diagonally (both down-right and down-left).\n- Vertical four is exactly a whole column (rows 0–3 in that column).\n- Diagonals of length 4 are exactly:\n  - Down-right (top-left → bottom-right): start only at row 0, columns 0–5. Each is (0,c),(1,c+1),(2,c+2),(3,c+3).\n  - Down-left (top-right → bottom-left): start only at row 0, columns 3–8. Each is (0,c),(1,c-1),(2,c-2),(3,c-3).\n\nAccepted coordinate inputs and normalization:\n- Accept these explicit coordinate forms:\n  - “(r,c)”\n  - “row r col c”\n  - “m r c” (move notation; treat exactly as r,c)\n- Accept explicit enumerations and expand to individual coordinates:\n  - “row 2 cols 6,7,8” → add "2,6", "2,7", "2,8"\n  - “column 5 rows 0,1,3” → add "0,5", "1,5", "3,5"\n  - “row 1 col 2” (ingle), “row 1 col 1 or col 0” → add "1,1" and "1,0"\n- Normalize every coordinate to the exact string "r,c" (no spaces).\n- Unambiguous-only: Ignore vague phrases (e.g., “left side”, “near the center”, “the other end”) unless the exact endpoint squares were enumerated earlier in that same branch context.\n\snRequired output format (strict JSON):\n- Produce exactly: {"trees": Node[]}\n- Node := ["r,c", Node, Node, ...]\n  - First element is the move coordinate string "r,c".\n  - Following elements (if any) are child Node values.\n- A leaf node is just ["r,c"].\n- Strict JSON only:\n  - Use double quotes for all strings.\n  - No comments, no trailing commas, no extra fields.\n  - Do not wrap your final JSON in Markdown/code fences.\n\nTree semantics and strict alternation:\n- Each root node is a model-side first move explicitly considered in the trace.\n- Depth alternates by side:\n  - Depth 0 (root): model move\n  - Depth 1: opponent reply\n  - Depth 2: model reply\n  - Depth 3: opponent reply\n  - … and so on. Maintain alternation at every depth.\n- Children under any node are exactly the explicitly described replies for that exact position/branch.\n\nHow to extract moves from the trace (precise, comprehensive):\n\n1) Identify model-side root moves (first-move candidates only)\n- Create a separate root node for every distinct coordinate the trace explicitly considers as the model’s first move.\n- Treat as model-first-move candidates when phrased as:\n  - “If I play (r,c)…”, “I can/could/should play (r,c)…”, “What if I play (r,c)…”\n  - “Another idea: (r,c)…”, “The best move is (r,c)…” (final chosen move still counts as a root)\n  - Enumerations: “I could play (a,b) or (c,d)” → add both as separate roots.\n  - Series: “I could fill (2,1),(2,2),(2,3)…” → add each as a separate root.\n  - “m r c” used to propose the model’s move choice → add as a root.\n- Include every distinct explicitly proposed model-first-move, even if later rejected or deemed inferior. Do not drop quick tests or briefly examined options.\n- Do NOT promote coordinates to roots if they appear only as opponent moves or only as deeper replies (unless the trace also proposes them as a model-first move elsewhere).\n\n2) Build subtrees per root (no cross-pollination across roots)\n- Under a specific root, attach opponent replies tied to that exact root scenario. Use phrasing such as: “then they can…”, “the opponent can reply at …”, “they must block at …”, “they’ll answer with …”, “they can try …”, “they target …”.\n- Include opponent move enumerations tied to that root, in the order explicitly mentioned:\n  - Column/row enumerations: e.g., “Their vertical in column 5 needs rows 0,1,3” → add "0,5","1,5","3,5".\n  - Horizontal targets: “After (0,3), they threaten row 2 cols 6,7,8” → add "2,6","2,7","2,8".\n  - Diagonal endpoints: If a diagonal is described or implied by endpoints (e.g., “(0,3)-(1,4)-(2,5)-(3,6)” or “they could play (0,3) or (3,6)”), add each explicitly named endpoint as an opponent child.\n- For each opponent child, attach exactly the model’s explicitly stated reply(ies) for that branch, maintaining alternation. If multiple model replies are given (“I must play (2,2) or (2,6)”), add both as children under that opponent node.\n- Continue deeper when the trace specifies the next opponent move after the model reply (e.g., “after I block at (2,2), they win by the other end (2,6)”), adding that opponent move as the next child node at the correct depth.\n- Only include moves explicitly tied to the current root context. Do NOT transfer opponent replies or deeper sequences from one root to another unless the trace explicitly restates them for that other root.\n\n3) Mirroring, “if instead/similarly”, and “the other end” within the same root\n- If the trace presents mirrored alternatives (“or instead at (c,d)”, “similarly at (x,y)”), include each as a sibling child under the same parent, in order of mention.\n- “The other end” is allowed only if both endpoints of the line were explicitly established earlier in that branch. Add the mirrored endpoint if explicitly identified or implied by previously listed endpoints of that same line.\n- When both ends of a horizontal or diagonal threat are explicitly named, include both as children.\n\n4) Context tracking and deeper sequences\n- Maintain the exact branch context as described. When the trace walks through a sequence (“If I play (0,1), they must play (3,4); then I play (0,5); then they must block (3,2); then I can play (2,5) …”), build the path depth-by-depth under the same root with strict alternation.\n- If the narrative continues with “after that”, “then”, “from there”, or refers back to a previously described branch (without switching to a new root), continue that exact subtree.\n- Do not assume or create branches not explicitly stated.\n\n5) Unambiguous-only policy\n- Include only coordinates that are explicitly specified (via accepted formats or explicit enumerations).\n- Ignore vague references (“block there”, “left side”, “near (r,c)”) unless the exact squares were enumerated earlier in that same branch.\n\n6) De-duplication and ordering\n- Within the same parent, include each child coordinate at most once.\n- If a node reappears later under the same parent, merge newly stated children into that existing node (preserve alternation).\n- Preserve the order of children exactly as they are mentioned in the trace.\n\n7) Interpreting “m r c” notation\n- Treat “m r c” exactly as the explicit coordinate (r,c) and normalize to "r,c".\n- Add it as a root only if used to propose the model’s first move (including the final chosen move).\n- When “m r c” appears as an opponent reply in-branch, add it at the appropriate depth under the correct parent.\n\nGeneralizable extraction strategy (to avoid common mistakes seen in prior attempts):\n- Be exhaustive in capturing roots: every time the narrator explicitly proposes a candidate for “my first move” (including quick tests, alternatives, and series), create a root for that coordinate. Do not skip less-preferred or rejected first-move candidates.\n- Under each root, be exhaustive in capturing opponent replies explicitly tied to that root, including:\n  - All row/column enumerations, expanded to individual squares.\n  - Both explicitly named endpoints of diagonals/horizontals.\n  - Any “they could play/try/need/must” squares enumerated as part of threats on that branch.\n- Keep branches separate by root; do not mix children across roots.\n- Maintain strict side alternation at all depths.\n- Do not invent nodes; only add squares explicitly named or enumerated under the current branch context.\n- Do not promote opponent-only or deeper-branch-only squares to roots unless they were also explicitly presented as model-first-move options elsewhere in the trace.\n\nFinal validation checklist before output:\n- Top-level object is exactly {"trees": [...]} with Node[] content.\n- Every node is ["r,c", ...children...] with coordinates normalized to "r,c" (no spaces).\n- Root nodes include every distinct explicit model-first-move candidate from the trace (including all enumerated candidates and quick tests), in order of first mention.\n- Under each root, all explicitly stated opponent replies for that root are present (including all expanded row/column enumerations and diagonal/horizontal endpoints), in order of first mention.\n- Under each opponent node, all explicitly stated model replies are present; include deeper opponent follow-ups when specified.\n- No duplicates at the same parent; merge re-mentioned nodes’ children; preserve mention order.\n- Strict alternation by depth is preserved everywhere.\n- Strict JSON only; use double quotes; no trailing commas; no extra fields; do not wrap in Markdown.'
    trace = Field(annotation=str required=True json_schema_extra={'desc': "the model's reasoning trace", '__dspy_field_type': 'input', 'prefix': 'Trace:'})
    trees = Field(annotation=List[List[Any]] required=True json_schema_extra={'desc': 'JSON list-of-lists per Node spec above', '__dspy_field_type': 'output', 'prefix': 'Trees:'})
))

In [8]:
optimized_instructions = loaded_optimized_program.extract_move_trees_module.signature.instructions
dspy_user_prompt_stub = """[[ ## trace ## ]]
{trace}

Respond with a JSON object in the following order of fields: trees (must be formatted as a valid Python list[list[Any]])."""

upload to openai batch

In [9]:
from openai import OpenAI
from collections import Counter

In [10]:
client = OpenAI(timeout=900)

In [ ]:
jobs = []
for i, row in enumerate(dataset[5100:]):
    r = client.responses.create(
        model="gpt-5-2025-08-07",
        instructions=optimized_instructions,
        input=dspy_user_prompt_stub.format(trace=row["trace"]),
        temperature=1.0,
        max_output_tokens=32000,
        reasoning={"effort": "medium", "summary": "detailed"},
        # service_tier="flex",
        background=True,
        metadata={"game_path": row["game_path"], "turn_idx": str(row["turn_idx"])},
    )
    jobs.append({"response_id": r.id, "game_path": row["game_path"], "turn_idx": row["turn_idx"]})

with open("responses_jobs.jsonl", "w", encoding="utf-8") as f:
    for j in jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")

In [ ]:
counts = Counter()

for j in jobs:
    try:
        r = client.responses.retrieve(j["response_id"])
        s = getattr(r, "status", None) or (isinstance(r, dict) and r.get("status")) or "unknown"
        if s == "canceled": s = "cancelled"
        counts[s] += 1
    except Exception:
        counts["exception"] += 1

total = len(jobs)
finished = sum(counts[s] for s in ("completed","failed","cancelled","incomplete"))
errors = counts["failed"] + counts["cancelled"] + counts["incomplete"]

print(f"{finished}/{total} finished {errors} errors")
for s in ("queued","in_progress","completed","failed","cancelled","incomplete","exception","unknown"):
    if counts.get(s): print(f"{s}: {counts[s]}")

fix failed ones

In [ ]:
with open("failed_response.jsonl", "r") as f:
    failed = [json.loads(ln) for ln in f]

In [ ]:
retry_jobs = []
for i, row in enumerate(dataset):
    if (row["game_path"], row["turn_idx"]) in zip(map(lambda x: x["game_path"], failed), map(lambda x: x["turn_idx"], failed)):
        # print(row)
        r = client.responses.create(
            model="gpt-5-2025-08-07",
            instructions=optimized_instructions,
            input=dspy_user_prompt_stub.format(trace=row["trace"]),
            temperature=1.0,
            max_output_tokens=32000,
            reasoning={"effort": "medium", "summary": "detailed"},
            # service_tier="flex",
            # background=True,
            metadata={"game_path": row["game_path"], "turn_idx": str(row["turn_idx"])},
        )
        retry_jobs.append({"response_id": r.id, "game_path": row["game_path"], "turn_idx": row["turn_idx"]})

with open("retry_jobs.jsonl", "w", encoding="utf-8") as f:
    for j in retry_jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")

In [ ]:
counts = Counter()

for j in retry_jobs:
    try:
        r = client.responses.retrieve(j["response_id"])
        s = getattr(r, "status", None) or (isinstance(r, dict) and r.get("status")) or "unknown"
        if s == "canceled": s = "cancelled"
        counts[s] += 1
    except Exception:
        counts["exception"] += 1
        

total = len(retry_jobs)
finished = sum(counts[s] for s in ("completed","failed","cancelled","incomplete"))
errors = counts["failed"] + counts["cancelled"] + counts["incomplete"]

print(f"{finished}/{total} finished {errors} errors")
for s in ("queued","in_progress","completed","failed","cancelled","incomplete","exception","unknown"):
    if counts.get(s): print(f"{s}: {counts[s]}")

add kimi-k2-thinking

In [ ]:
jobs = []
for i, row in enumerate(dataset):
    if "kimi-k2-thinking" in row["game_path"]:
        r = client.responses.create(
            model="gpt-5-2025-08-07",
            instructions=optimized_instructions,
            input=dspy_user_prompt_stub.format(trace=row["trace"]),
            temperature=1.0,
            max_output_tokens=32000,
            reasoning={"effort": "medium", "summary": "detailed"},
            # service_tier="flex",
            background=True,
            metadata={"game_path": row["game_path"], "turn_idx": str(row["turn_idx"])},
        )
        jobs.append({"response_id": r.id, "game_path": row["game_path"], "turn_idx": row["turn_idx"]})

with open("responses_jobs_add_kimi_k2_thinking.jsonl", "w", encoding="utf-8") as f:
    for j in jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")

In [14]:
counts = Counter()

for j in jobs:
    try:
        r = client.responses.retrieve(j["response_id"])
        s = getattr(r, "status", None) or (isinstance(r, dict) and r.get("status")) or "unknown"
        if s == "canceled": s = "cancelled"
        counts[s] += 1
    except Exception:
        counts["exception"] += 1

total = len(jobs)
finished = sum(counts[s] for s in ("completed","failed","cancelled","incomplete"))
errors = counts["failed"] + counts["cancelled"] + counts["incomplete"]

print(f"{finished}/{total} finished {errors} errors")
for s in ("queued","in_progress","completed","failed","cancelled","incomplete","exception","unknown"):
    if counts.get(s): print(f"{s}: {counts[s]}")

1049/1049 finished 4 errors
completed: 1045
failed: 4


finished all runs, test if they all have correct jsons

In [1]:
from json_repair import repair_json
from tqdm import tqdm
from pathlib import Path
import json

In [2]:
results = []
with open("/Users/saner/game-arena/game-arena/notebooks/succeeded.jsonl", "r") as f:
    for line in f:
        results.append(json.loads(line))

In [3]:
base_path = Path("/Users/saner/game-arena/game-arena/aws-logs/final/four_in_a_row-fen/")

for idx, result in tqdm(enumerate(results)):
    response = result["response"]["output"][-1]
    if response["type"] != "message":
        print("Non message final output item, ID:", result["response_id"], result["game_path"], result["turn_idx"])
    else:
        repaired = repair_json(response["content"][-1]["text"])
        j = json.loads(repaired)

        with open(base_path / result["game_path"], "r") as f:
            game_log = json.load(f)
            
        
        results[idx]["model"] = game_log["players"][game_log["events"][result["turn_idx"]]["player"]]
        results[idx]["trees"] = j["trees"]
        results[idx]["move_success"] = game_log["events"][result["turn_idx"] + 1]["type"] == "move_success" 
        results[idx]["extracted_move"] = game_log["events"][result["turn_idx"]]["extracted_move"]

0it [00:00, ?it/s]

18888it [00:07, 2554.18it/s]


In [4]:
len(results)

18888

In [5]:
import pandas as pd

def records_to_df(records):
    """
    Convert iterable of records (dicts shaped like your sample) into a DataFrame
    with columns: game_path, turn_idx, model, trees.
    - Safely handles missing keys.
    - Preserves `trees` as a Python object (list) for downstream feature functions.
    """
    rows = []
    for rec in records:
        game_path = rec.get("game_path")
        turn_idx = rec.get("turn_idx")
        model = rec.get("model")
        trees = rec.get("trees")
        move_success = rec.get("move_success")
        extracted_move = rec.get("extracted_move")
        judge_response = rec.get("response")
        rows.append(
            {
                "game_path": game_path,
                "turn_idx": turn_idx,
                "model": model,
                "trees": trees,
                "extracted_move": extracted_move,
                "move_success": move_success,
                "judge_response": judge_response
            }
        )
    df = pd.DataFrame(rows, columns=["game_path", "turn_idx", "model", "trees", "extracted_move", "move_success"]) # "judge_response"
    return df

df = records_to_df(results)

In [6]:
# df.loc[18886]["judge_response"]

derived features

In [7]:
from dataclasses import dataclass
from collections import defaultdict
from typing import Any, Dict, List, Tuple, Iterable, Optional
import math
import pandas as pd

def _children(node: Any) -> List[Any]:
    """Return child nodes for either ['r,c', ...] or [r, c, ...]."""
    if not isinstance(node, (list, tuple)) or not node:
        return []
    first = node[0]
    if isinstance(first, str):
        return [ch for ch in node[1:] if isinstance(ch, (list, tuple))]
    else:
        return [ch for ch in node[2:] if isinstance(ch, (list, tuple))]

@dataclass
class ForestCounts:
    max_depth: int
    n_roots: int
    n_leaves: int
    sum_leaf_depths: float
    total_nodes: int
    internal_nodes: int
    expansions: int
    # depth → node count at that depth (root depth = 1)
    level_counts: Dict[int, int]
    # for effort variants
    sum_node_depths: float
    sum_node_depths_p: float  # depth^p (p chosen at collection time)

def collect_counts(
    trees: Any,
    p_for_effort: float = 1.0,
) -> ForestCounts:
    """
    Single DFS pass that gathers everything needed for all metrics.
    Root depth = 1. Returns zeros for empty/malformed trees.
    """
    if not isinstance(trees, (list, tuple)) or len(trees) == 0:
        return ForestCounts(
            max_depth=0, n_roots=0, n_leaves=0, sum_leaf_depths=0.0,
            total_nodes=0, internal_nodes=0, expansions=0,
            level_counts=defaultdict(int),
            sum_node_depths=0.0, sum_node_depths_p=0.0
        )

    stack: List[Tuple[Any, int]] = [(t, 1) for t in trees if isinstance(t, (list, tuple))]
    max_depth = 0
    n_leaves = 0
    sum_leaf_depths = 0.0
    total_nodes = 0
    internal_nodes = 0
    expansions = 0
    level_counts = defaultdict(int)
    sum_node_depths = 0.0
    sum_node_depths_p = 0.0

    while stack:
        node, depth = stack.pop()
        total_nodes += 1
        level_counts[depth] += 1
        max_depth = max(max_depth, depth)

        sum_node_depths += depth
        sum_node_depths_p += depth ** p_for_effort

        kids = _children(node)
        if kids:
            internal_nodes += 1
            expansions += len(kids)
            for ch in kids:
                stack.append((ch, depth + 1))
        else:
            n_leaves += 1
            sum_leaf_depths += depth

    return ForestCounts(
        max_depth=max_depth,
        n_roots=len(trees),
        n_leaves=n_leaves,
        sum_leaf_depths=sum_leaf_depths,
        total_nodes=total_nodes,
        internal_nodes=internal_nodes,
        expansions=expansions,
        level_counts=level_counts,
        sum_node_depths=sum_node_depths,
        sum_node_depths_p=sum_node_depths_p,
    )

def max_ply(cnt: ForestCounts) -> int: #
    return int(cnt.max_depth)

def n_root_moves(cnt: ForestCounts) -> int:
    return int(cnt.n_roots)

def n_leaf_moves(cnt: ForestCounts) -> int:
    return int(cnt.n_leaves)

def plan_area(cnt: ForestCounts) -> float:
    """Sum of leaf depths (breadth × depth)."""
    return float(cnt.sum_leaf_depths)

def mean_ply(cnt: ForestCounts) -> float:
    return float(cnt.sum_leaf_depths / cnt.n_leaves) if cnt.n_leaves else 0.0

def total_nodes(cnt: ForestCounts) -> int:
    return int(cnt.total_nodes)

def expansions(cnt: ForestCounts) -> int:
    """Total edges explored."""
    return int(cnt.expansions)

def avg_branching(cnt: ForestCounts) -> float:
    return float(cnt.expansions / cnt.internal_nodes) if cnt.internal_nodes else 0.0

def ebf(cnt: ForestCounts) -> float:
    """
    Effective branching factor = geometric mean over depth of (#nodes_{d+1} / #nodes_d).
    """
    ratios = []
    for d in range(1, cnt.max_depth):
        nd = cnt.level_counts.get(d, 0)
        nd1 = cnt.level_counts.get(d + 1, 0)
        if nd > 0 and nd1 > 0:
            ratios.append(nd1 / nd)
    if not ratios:
        return 0.0
    return math.exp(sum(math.log(r) for r in ratios) / len(ratios))

def search_effort(cnt: ForestCounts, p_used: float) -> float:
    """
    Depth-weighted node count (Σ depth^p across all nodes).
    NOTE: p_used must match the p_for_effort used in collect_counts.
    """
    return float(cnt.sum_node_depths_p if p_used != 1.0 else cnt.sum_node_depths)


def compute_metrics(trees: Any, p_for_effort: float = 1.5) -> Dict[str, float]:
    """
    One-call convenience: collect counts once, then compute all metrics.
    """
    cnt = collect_counts(trees, p_for_effort)
    return {
        "max_ply": max_ply(cnt),
        "n_root_moves": n_root_moves(cnt),
        "n_leaf_moves": n_leaf_moves(cnt),
        "plan_area": plan_area(cnt),
        "mean_ply": mean_ply(cnt),
        "total_nodes": total_nodes(cnt),
        "expansions": expansions(cnt),
        "avg_branching": avg_branching(cnt),
        "ebf": ebf(cnt),
        "search_effort": search_effort(cnt, p_for_effort),
        "effort_p": float(p_for_effort),
    }

def add_metrics_columns(df: pd.DataFrame, trees_col: str = "trees", p_for_effort: float = 1.5) -> pd.DataFrame:
    """
    Applies compute_metrics to each row's forest and returns df with new metric columns.
    """
    metrics_df = df[trees_col].apply(lambda t: compute_metrics(t, p_for_effort)).apply(pd.Series)
    return pd.concat([df, metrics_df], axis=1)


In [8]:
df = add_metrics_columns(df)

In [9]:
df

,game_path,turn_idx,model,trees,extracted_move,move_success,max_ply,n_root_moves,n_leaf_moves,plan_area,mean_ply,total_nodes,expansions,avg_branching,ebf,search_effort,effort_p
0,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,12,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[0,5, [0,6]], [1,3], [1,5]]",m 0 5,True,2.0,3.0,3.0,4.0,1.333333,4.0,1.0,1.0,0.333333,5.828427,1.5
1,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,14,"{'provider': 'vertex', 'model': 'qwen/qwen3-ne...","[[3,6], [0,6], [1,3], [3,5]]",m 0 6,True,1.0,4.0,4.0,4.0,1.000000,4.0,0.0,0.0,0.000000,4.000000,1.5
2,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,16,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[1,3], [1,5], [2,4]]",m 1 3,True,1.0,3.0,3.0,3.0,1.000000,3.0,0.0,0.0,0.000000,3.000000,1.5
3,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,18,"{'provider': 'vertex', 'model': 'qwen/qwen3-ne...","[[3,6], [3,5], [3,4], [1,5], [3,3, [2,3]], [0,...",m 2 3,True,2.0,13.0,13.0,15.0,1.153846,15.0,2.0,1.0,0.153846,18.656854,1.5
4,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,20,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[2,2, [3,1]], [3,1]]",m 2 2,True,2.0,2.0,2.0,3.0,1.500000,3.0,1.0,1.0,0.500000,4.828427,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18883,kimi-k2-thinking_vs_deepseek-v3.1-thinking-ver...,20,"{'provider': 'vertex', 'model': 'deepseek-ai/d...","[[0,1]]",m 0 1,True,1.0,1.0,1.0,1.0,1.000000,1.0,0.0,0.0,0.000000,1.000000,1.5
18884,kimi-k2-thinking_vs_kimi-k2-0905/batch-e545772...,24,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 2 5,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5
18885,kimi-k2-thinking_vs_qwen3-max/batch-e545772d-a...,16,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 1 2,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5
18886,kimi-k2-thinking_vs_gemini-2.5-pro/batch-e5457...,12,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 0 3,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5


add model names

model name aliasing

In [10]:
# Model name mapping: new model field -> old model field (from models_config.yaml)
# Maps actual model values, not config keys
MODEL_NAME_ALIASES = {
    # DeepSeek models: vertex -> fireworks
    'deepseek-ai/deepseek-v3.1-maas': 'accounts/fireworks/models/deepseek-v3p1',
    'deepseek-ai/deepseek-r1-0528-maas': 'accounts/fireworks/models/deepseek-r1-0528',
    
    # Claude models: vertex-anthropic -> anthropic
    'claude-opus-4-1@20250805': 'claude-opus-4-1-20250805',
    'claude-sonnet-4@20250514': 'claude-sonnet-4-20250514',
}


def get_legacy_model_name(model_id: str) -> str:
    """Get the legacy/old model name for backward compatibility.
    Converts new vertex names back to original names for storage."""
    return MODEL_NAME_ALIASES.get(model_id, model_id)


def normalize_model_name(model_id: str) -> str:
    """Normalize model names to use legacy names for backward compatibility.
    This ensures all storage (S3, DynamoDB) uses consistent old names."""
    return get_legacy_model_name(model_id)

def get_model_identifier(player_data) -> str:
    """Get model identifier including reasoning effort if present.
    Normalizes to legacy names for backward compatibility.
    Handles special cases like deepseek-v3.1 thinking config."""
    model_name = player_data.get('model', '')
    config = player_data.get('config', {})
    reasoning_effort = config.get('reasoning_effort', '')
    
    # Special case: deepseek-v3.1 with thinking config
    # Only the thinking:true version maps to old model
    if model_name == 'deepseek-ai/deepseek-v3.1-maas':
        thinking = config.get('thinking', False)
        if thinking:
            # This is the old thinking version - map to fireworks
            normalized_model = 'accounts/fireworks/models/deepseek-v3p1'
            # Old version had reasoning_effort: medium
            return f"{normalized_model}-medium"
        else:
            # This is the NEW non-thinking version - keep as-is (no mapping)
            return model_name
    
    # Special case: deepseek-r1 vertex always maps to old with -medium
    # Old fireworks version had reasoning_effort: medium
    if model_name == 'deepseek-ai/deepseek-r1-0528-maas':
        normalized_model = 'accounts/fireworks/models/deepseek-r1-0528'
        return f"{normalized_model}-medium"
    
    # Normalize base model name first
    normalized_model = normalize_model_name(model_name)
    
    # Add reasoning effort to normalized name if present
    if reasoning_effort:
        return f"{normalized_model}-{reasoning_effort}"
    
    return normalized_model

In [11]:
import pandas as pd

# --- Aliases & helpers you provided ---
MODEL_NAME_ALIASES = {
    # DeepSeek models: vertex -> fireworks
    'deepseek-ai/deepseek-v3.1-maas': 'accounts/fireworks/models/deepseek-v3p1',
    'deepseek-ai/deepseek-r1-0528-maas': 'accounts/fireworks/models/deepseek-r1-0528',

    # Claude models: vertex-anthropic -> anthropic
    'claude-opus-4-1@20250805': 'claude-opus-4-1-20250805',
    'claude-sonnet-4@20250514': 'claude-sonnet-4-20250514',
}

def get_legacy_model_name(model_id: str) -> str:
    """Get the legacy/old model name for backward compatibility."""
    return MODEL_NAME_ALIASES.get(model_id, model_id)

def normalize_model_name(model_id: str) -> str:
    """Normalize model names to use legacy names for backward compatibility."""
    return get_legacy_model_name(model_id)

def get_model_identifier(player_data) -> str:
    """Get model identifier including reasoning effort if present.
    Normalizes to legacy names for backward compatibility.
    Handles special cases like deepseek-v3.1 thinking config."""
    model_name = player_data.get('model', '') or ''
    config = player_data.get('config', {}) or {}
    reasoning_effort = config.get('reasoning_effort', '')

    # Special case: deepseek-v3.1 with thinking config
    if model_name == 'deepseek-ai/deepseek-v3.1-maas':
        thinking = config.get('thinking', False)
        if thinking:
            # Old thinking version
            normalized_model = 'accounts/fireworks/models/deepseek-v3p1'
            return f"{normalized_model}-medium"
        else:
            # NEW non-thinking version - keep as-is
            return model_name

    # Special case: deepseek-r1 vertex always maps to old with -medium
    if model_name == 'deepseek-ai/deepseek-r1-0528-maas':
        normalized_model = 'accounts/fireworks/models/deepseek-r1-0528'
        return f"{normalized_model}-medium"

    # Default: normalize + add -<reasoning_effort> if present
    normalized_model = normalize_model_name(model_name)
    if reasoning_effort:
        # avoid duplicate suffix (e.g., already endswith '-medium')
        if not str(normalized_model).endswith(f"-{reasoning_effort}"):
            return f"{normalized_model}-{reasoning_effort}"
    return normalized_model


# --- Replacement for the YAML-based function ---
def add_model_names_from_yaml_strict(df: pd.DataFrame,
                                     yaml_path=None,  # kept for compatibility; ignored
                                     model_col: str = "model",
                                     out_col: str = "model_names") -> pd.DataFrame:
    """
    Compute friendly model names for each row using alias mappings and config,
    instead of reading from models_config.yaml.

    - If `model_col` cell is a dict, we look for:
        * 'model' (string) OR 'model' (dict with 'name'/'id') OR top-level 'name'/'id'
        * optional 'config' dict (may include 'reasoning_effort' and 'thinking')
      Then we apply get_model_identifier() to produce the final name.
    - If it's a string, we normalize via aliases; if reasoning info isn't available,
      we just return the normalized base name.
    - If nothing usable is found, we raise with the offending rows.
    """
    def _extract_player_data(cell):
        # Returns {'model': <str>, 'config': <dict>} or None
        if isinstance(cell, dict):
            cfg = cell.get('config', {}) or {}

            # Hoist top-level reasoning/thinking into config if present
            if 'reasoning_effort' in cell and 'reasoning_effort' not in cfg:
                cfg = {**cfg, 'reasoning_effort': cell['reasoning_effort']}
            if 'thinking' in cell and 'thinking' not in cfg:
                cfg = {**cfg, 'thinking': cell['thinking']}

            # Common shapes:
            # 1) {'provider': ..., 'model': 'name', 'config': {...}}
            # 2) {'model': {'name': 'name', 'id': '...'}, 'config': {...}}
            # 3) {'name': 'name', ...} or {'id': '...'}
            model_val = cell.get('model')
            model_name = None
            if isinstance(model_val, str):
                model_name = model_val
            elif isinstance(model_val, dict):
                model_name = model_val.get('name') or model_val.get('id') or model_val.get('model')
            # Fallbacks
            if not model_name:
                model_name = cell.get('name') or cell.get('id')

            if model_name:
                return {'model': model_name, 'config': cfg}
            return None

        if isinstance(cell, str):
            return {'model': cell, 'config': {}}

        return None

    def _resolve(cell):
        pdict = _extract_player_data(cell)
        if not pdict:
            return None
        try:
            return get_model_identifier(pdict)
        except Exception:
            # Very defensive fallback: base name + optional suffix if we can see it
            base = normalize_model_name(pdict.get('model') or '')
            if not base:
                return None
            reff = (pdict.get('config') or {}).get('reasoning_effort')
            if reff and not base.endswith(f"-{reff}"):
                return f"{base}-{reff}"
            return base

    names = df[model_col].apply(_resolve)

    # Enforce complete mapping if anything is still missing
    if not names.notna().all():
        bad_rows = df.index[~names.notna()].tolist()

        def _fmt(x):
            if isinstance(x, dict):
                # show a compact summary for debugging
                model_val = x.get('model')
                if isinstance(model_val, dict):
                    model_val = model_val.get('name') or model_val.get('id') or model_val
                return {
                    "model": model_val or x.get('name') or x.get('id'),
                    "config": x.get('config') or {},
                }
            return x

        bad_vals = df.loc[bad_rows, model_col].apply(_fmt).tolist()
        raise ValueError(f"Unmapped/invalid model rows: {bad_rows} -> {bad_vals}")

    return df.assign(**{out_col: names})

In [12]:
df = add_model_names_from_yaml_strict(df, yaml_path="../models_config.yaml")

In [13]:
df

,game_path,turn_idx,model,trees,extracted_move,move_success,max_ply,n_root_moves,n_leaf_moves,plan_area,mean_ply,total_nodes,expansions,avg_branching,ebf,search_effort,effort_p,model_names
0,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,12,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[0,5, [0,6]], [1,3], [1,5]]",m 0 5,True,2.0,3.0,3.0,4.0,1.333333,4.0,1.0,1.0,0.333333,5.828427,1.5,gpt-5-mini-2025-08-07-high
1,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,14,"{'provider': 'vertex', 'model': 'qwen/qwen3-ne...","[[3,6], [0,6], [1,3], [3,5]]",m 0 6,True,1.0,4.0,4.0,4.0,1.000000,4.0,0.0,0.0,0.000000,4.000000,1.5,qwen/qwen3-next-80b-a3b-thinking-maas
2,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,16,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[1,3], [1,5], [2,4]]",m 1 3,True,1.0,3.0,3.0,3.0,1.000000,3.0,0.0,0.0,0.000000,3.000000,1.5,gpt-5-mini-2025-08-07-high
3,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,18,"{'provider': 'vertex', 'model': 'qwen/qwen3-ne...","[[3,6], [3,5], [3,4], [1,5], [3,3, [2,3]], [0,...",m 2 3,True,2.0,13.0,13.0,15.0,1.153846,15.0,2.0,1.0,0.153846,18.656854,1.5,qwen/qwen3-next-80b-a3b-thinking-maas
4,gpt-5-mini-high_vs_qwen3-next-thinking-vertex/...,20,"{'provider': 'openai', 'model': 'gpt-5-mini-20...","[[2,2, [3,1]], [3,1]]",m 2 2,True,2.0,2.0,2.0,3.0,1.500000,3.0,1.0,1.0,0.500000,4.828427,1.5,gpt-5-mini-2025-08-07-high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18883,kimi-k2-thinking_vs_deepseek-v3.1-thinking-ver...,20,"{'provider': 'vertex', 'model': 'deepseek-ai/d...","[[0,1]]",m 0 1,True,1.0,1.0,1.0,1.0,1.000000,1.0,0.0,0.0,0.000000,1.000000,1.5,accounts/fireworks/models/deepseek-v3p1-medium
18884,kimi-k2-thinking_vs_kimi-k2-0905/batch-e545772...,24,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 2 5,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5,kimi-k2-thinking
18885,kimi-k2-thinking_vs_qwen3-max/batch-e545772d-a...,16,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 1 2,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5,kimi-k2-thinking
18886,kimi-k2-thinking_vs_gemini-2.5-pro/batch-e5457...,12,"{'provider': 'moonshot', 'model': 'kimi-k2-thi...",[],m 0 3,True,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.5,kimi-k2-thinking


In [14]:
col = 'model_names' if 'model_names' in df.columns else 'model_name'
for m in sorted(df[col].dropna().astype(str).unique()):
    print(m)

accounts/fireworks/models/deepseek-r1-0528-medium
accounts/fireworks/models/deepseek-v3p1-medium
accounts/fireworks/models/glm-4p5-medium
accounts/fireworks/models/kimi-k2-instruct-0905
accounts/fireworks/models/llama-v3p3-70b-instruct
accounts/fireworks/models/qwen3-235b-a22b-thinking-2507-medium
claude-opus-4-1-20250805
claude-sonnet-4-20250514
claude-sonnet-4-5@20250929
deepseek-ai/deepseek-v3.1-maas
deepseek-reasoner
gemini-2.5-pro
glm-4.6
gpt-4o-2024-08-06
gpt-5-2025-08-07-high
gpt-5-2025-08-07-medium
gpt-5-mini-2025-08-07-high
gpt-5-mini-2025-08-07-medium
grok-4
grok-4-fast-reasoning
grok-code-fast-1
kimi-k2-thinking
o3-2025-04-16-high
openai/gpt-oss-120b-maas-high
openai/gpt-oss-120b-maas-medium
qwen/qwen3-next-80b-a3b-thinking-maas
qwen3-max


In [15]:
len(df)

18888

In [16]:
df.to_pickle("game_trees_df.pkl")

filtering for one model

In [17]:
mask = (
    df["model_names"].apply(lambda m: "4o" in m)
)

filtered = df[mask]

In [18]:
filtered

,game_path,turn_idx,model,trees,extracted_move,move_success,max_ply,n_root_moves,n_leaf_moves,plan_area,mean_ply,total_nodes,expansions,avg_branching,ebf,search_effort,effort_p,model_names
71,gpt-4o_vs_gpt-oss-120b-high-vertex/batch-1c821...,0,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[1,4]]",m 1 4,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
73,gpt-4o_vs_gpt-oss-120b-high-vertex/batch-1c821...,4,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[2,4]]",m 2 4,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
75,gpt-4o_vs_gpt-oss-120b-high-vertex/batch-1c821...,8,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[3,4]]",m 3 4,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
77,gpt-4o_vs_gpt-oss-120b-high-vertex/batch-1c821...,12,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[2,0]]",m 2 0,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
79,gpt-4o_vs_gpt-oss-120b-high-vertex/batch-1c821...,16,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[0,4]]",m 0 4,False,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18253,kimi-k2-thinking_vs_gpt-4o/batch-e545772d-a9dc...,8,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[1,4]]",m 1 4,False,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
18254,kimi-k2-thinking_vs_gpt-4o/batch-e545772d-a9dc...,10,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[1,3]]",m 1 3,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
18256,kimi-k2-thinking_vs_gpt-4o/batch-e545772d-a9dc...,14,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[0,4]]",m 0 4,True,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06
18258,kimi-k2-thinking_vs_gpt-4o/batch-e545772d-a9dc...,18,"{'provider': 'openai', 'model': 'gpt-4o-2024-0...","[[0,5]]",m 0 5,False,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.5,gpt-4o-2024-08-06


In [19]:
len(filtered)

609

In [20]:
sum(filtered.move_success == False) / len(filtered)

0.2315270935960591

normalize pkl

In [22]:
# One-cell normalizer for malformed Four-in-a-Row forests
# - Loads game_trees_df.pkl
# - Normalizes every row's `trees` into a proper forest of Nodes:
#     Node := ["r,c", childNode, childNode, ...]
#   where each coordinate is a STRING "r,c" (negatives allowed, though illegal later)
# - Fixes cases like bare ints at the same level: 0,2, 0,6  -> ["0,2"], ["0,6"]
# - Accepts "m r c" strings and converts to "r,c"
# - Saves to game_trees_df.normalized.pkl and prints a brief summary

import re
import math
import pandas as pd
from tqdm import tqdm

INPUT_PKL  = "game_trees_df.pkl"
OUTPUT_PKL = "game_trees_df.normalized.pkl"

# Accept "r,c" and "m r c" — negatives allowed (engine will flag illegals later)
M_RC = re.compile(r"^\s*m\s*(-?\d+)\s+(-?\d+)\s*$", re.IGNORECASE)
R_C  = re.compile(r"^\s*'?(-?\d+)\s*,\s*(-?\d+)'?\s*$")

def _coord_str(label: str) -> str:
    """
    Normalize a coordinate-like string to 'r,c'.
    Accepts 'r,c' (extra spaces/quotes ok) and 'm r c'.
    Raises ValueError if not parseable.
    """
    if isinstance(label, tuple) and label:
        label = label[0]
    if not isinstance(label, str):
        raise ValueError(f"coord_label_not_str:{type(label).__name__}")
    m = R_C.match(label)
    if m:
        return f"{int(m.group(1))},{int(m.group(2))}"
    m = M_RC.match(label)
    if m:
        return f"{int(m.group(1))},{int(m.group(2))}"
    raise ValueError(f"coord_parse_failed:{label!r}")

def _is_well_formed_node(node) -> bool:
    """Strict check: node must be ['r,c', childNode, ...] with all children strict."""
    if not isinstance(node, (list, tuple)) or not node:
        return False
    head = node[0]
    if not isinstance(head, str):
        return False
    if not R_C.match(head):  # must already be 'r,c' string
        return False
    for ch in node[1:]:
        if not _is_well_formed_node(ch):
            return False
    return True

def _is_well_formed_forest(obj) -> bool:
    if obj == []:
        return True
    if not isinstance(obj, (list, tuple)):
        return False
    return all(_is_well_formed_node(x) for x in obj)

def _normalize_items(items):
    """
    Convert a heterogeneous list at a given level into a list of **node lists**.
    Rules:
      - consecutive int pairs r,c -> ['r,c']
      - coord strings -> ['r,c']
      - sublists/tuples -> normalized (node or forest)
      - dangling scalars are ignored
    """
    out = []
    i = 0
    n = len(items)
    while i < n:
        x = items[i]

        # 1) subtree/list/tuple
        if isinstance(x, (list, tuple)) and x:
            node_or_forest = normalize_to_forest(x)
            # normalize_to_forest always returns a forest (list of nodes)
            out.extend(node_or_forest)
            i += 1
            continue

        # 2) coordinate string
        if isinstance(x, str):
            try:
                out.append([_coord_str(x)])
            except ValueError:
                # ignore non-coord strings
                pass
            i += 1
            continue

        # 3) consecutive int pair -> node
        if isinstance(x, int) and (i + 1) < n and isinstance(items[i + 1], int):
            out.append([f"{int(x)},{int(items[i + 1])}"])
            i += 2
            continue

        # 4) dangling junk -> skip
        i += 1

    return out

def normalize_to_forest(obj):
    """
    Normalize ANY nested structure into a **forest** (list of nodes).
    A Node is always ['r,c', childNode, childNode, ...] after normalization.
    """
    # None/NaN -> empty forest
    if obj is None or (isinstance(obj, float) and math.isnan(obj)):
        return []

    # Node-like list/tuple?
    if isinstance(obj, (list, tuple)) and obj:
        head = obj[0]
        tail = list(obj[1:])

        # Case A: head is coordinate string or annotated tuple
        if isinstance(head, (str, tuple)):
            try:
                label = _coord_str(head)
                children = _normalize_items(tail)
                return [[label] + children]  # single node as a forest
            except ValueError:
                # Not a coord at head -> treat whole thing as a forest of items
                return _normalize_items(list(obj))

        # Case B: head is int and next is int => node head as int pair
        if isinstance(head, int) and len(obj) >= 2 and isinstance(obj[1], int):
            label = f"{int(obj[0])},{int(obj[1])}"
            children = _normalize_items(list(obj[2:]))
            return [[label] + children]  # single node as a forest

        # Otherwise treat this level as a forest of mixed items
        return _normalize_items(list(obj))

    # Single coordinate string
    if isinstance(obj, str):
        try:
            return [[ _coord_str(obj) ]]
        except ValueError:
            return []  # not a coord string -> drop

    # Anything else -> empty
    return []

# ------------------ run normalization ------------------

df = pd.read_pickle(INPUT_PKL)

orig = df["trees"]
norm = []
fixed_idx = []
errors = []

for i, trees in tqdm(list(orig.items()), total=len(orig), desc="Normalizing forests"):
    try:
        was_well_formed = _is_well_formed_forest(trees)
        normalized = normalize_to_forest(trees)
        norm.append(normalized)
        if not was_well_formed:
            fixed_idx.append(i)
    except Exception as e:
        # Worst case, fallback to empty forest (record error)
        norm.append([])
        fixed_idx.append(i)
        errors.append((i, str(e)))

df["trees"] = norm
df.to_pickle(OUTPUT_PKL)

print(f"\nSaved normalized DataFrame → {OUTPUT_PKL}")
print(f"Total rows: {len(df)}")
print(f"Rows that required normalization: {len(fixed_idx)}")
if errors:
    print(f"Rows with normalization errors (set to []): {len(errors)}")
    for i, msg in errors[:10]:
        print(f"  - row {i}: {msg}")

# Show a couple of before/after diffs
print("\nSample before/after (first 3 fixes):")
for i in fixed_idx[:3]:
    print(f"\nrow {i}:")
    print("BEFORE:", repr(orig.loc[i])[:800], "..." if len(repr(orig.loc[i])) > 800 else "")
    print("AFTER: ", repr(df.loc[i, "trees"])[:800], "..." if len(repr(df.loc[i, 'trees'])) > 800 else "")


Normalizing forests: 100%|██████████| 18888/18888 [00:00<00:00, 127210.74it/s]



Saved normalized DataFrame → game_trees_df.normalized.pkl
Total rows: 18888
Rows that required normalization: 0

Sample before/after (first 3 fixes):
